# Inicialização

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType, IntegerType
from pyspark.sql.functions import col, trim, to_date

# Carregar dados das tabelas Silver

In [0]:
query = """SELECT 
    ROW_NUMBER () OVER (ORDER BY crm_customers.customer_id) AS customer_key,
    crm_customers.customer_id,
    CONCAT(crm_customers.first_name, ' ', crm_customers.last_name) AS customer_name,
    crm_customers.marital_status,
    erp_customer_location.country,
    CASE WHEN crm_customers.gender = 'n/a'
    THEN erp_customers.gender
    ELSE crm_customers.gender
    END AS gender,
    erp_customers.birth_date as birth_date
FROM workspace.silver.crm_customers crm_customers
LEFT JOIN workspace.silver.erp_customers erp_customers
ON crm_customers.customer_key = erp_customers.customer_id
LEFT JOIN workspace.silver.erp_customer_location erp_customer_location
ON crm_customers.customer_key = erp_customer_location.customer_id;"""
df = spark.sql(query)
df.limit(100).display()


# Criação de tabela Gold

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_customers")

# Verificação final

In [0]:
%sql
SELECT * FROM workspace.gold.dim_customers LIMIT 1000;